<a href="https://colab.research.google.com/github/kennyaakin/Stable_Diffusion_VAE/blob/main/Stable_Diffusion_VAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle
!pip install denoising_diffusion_pytorch uformer-pytorch

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import torchvision.models as models
import math
import os
import shutil
from sklearn.model_selection import train_test_split
import os
from google.colab import drive
from denoising_diffusion_pytorch import Unet, GaussianDiffusion
from uformer_pytorch import Uformer
drive.mount('/content/drive')

In [ ]:
! mkdir ~/.kaggle

In [ ]:
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json

In [ ]:
! chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#! kaggle datasets download smaildurcan/turkish-license-plate-dataset
! kaggle datasets download tustunkok/synthetic-turkish-license-plates

In [ ]:
#! unzip turkish-license-plate-dataset.zip
! unzip synthetic-turkish-license-plates.zip

In [ ]:
'''# Comment this whole cell out when using the synthetic-turkish-license-plates dataset
# Set your dataset directory
image_dir = "/content/images"  # Replace this with the path where your images are

# Create a new directory for only jpg images (if it doesn't exist)
filtered_image_dir = "/content/filtered_images"
os.makedirs(filtered_image_dir, exist_ok=True)

# List all files in the directory
files = os.listdir(image_dir)

# Filter for only jpg files
for file in files:
    if file.lower().endswith('.jpg'):
        # Copy the .jpg files to the filtered directory
        shutil.copy(os.path.join(image_dir, file), os.path.join(filtered_image_dir, file))

print(f"Filtered .jpg files have been copied to: {filtered_image_dir}")
'''

In [ ]:
'''def split_dataset(source_dir, train_dir, test_dir, test_size=0.8, random_state=42):
    image_files = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    train_files, test_files = train_test_split(image_files, test_size=test_size, random_state=random_state)

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    for file in train_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(train_dir, file))

    for file in test_files:
        shutil.copy(os.path.join(source_dir, file), os.path.join(test_dir, file))

    print(f"Dataset split complete. {len(train_files)} training images, {len(test_files)} test images.")

#source_dir = "./filtered_images" #Comment this when using the turkish-license-plate-dataset
source_dir = "./license-plates" #Uncomment this when using the synthetic-turkish-license-plates dataset
train_dir = "./data/train/images"
test_dir = "./data/test/images"

split_dataset(source_dir, train_dir, test_dir)'''

import os, shutil

root = "./license-plates"
os.makedirs(os.path.join(root, "plates"), exist_ok=True)

for fname in os.listdir(root):
    if fname.lower().endswith((".jpeg", ".jpg", ".png")):
        shutil.move(os.path.join(root, fname), os.path.join(root, "plates", fname))

In [ ]:
def validate(model, val_loader):
    model.eval()
    total_loss, count = 0.0, 0
    with torch.no_grad():
        for imgs, _ in val_loader:
            imgs = imgs.to(device)
            loss = model(imgs).mean()
            total_loss += loss.item()
            count += 1
    return total_loss / count

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms
import gc
#from torch.utils.data import random_split
# from model import Encoder, Decoder

# Device configuration
device = torch.device('cuda')

# Hyperparameters
num_epochs = 10
learning_rate = 1e-4
beta = 0.00025  # KL divergence weight

# Data loading
transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
batch_size = 32
dataset = torchvision.datasets.ImageFolder(root='./license-plates', transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)


#model = VAE().to(device)
unet = Unet(dim=96, dim_mults=(1,2,4,8))
model = GaussianDiffusion(unet, image_size=96, timesteps=1000).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Add these hyperparameters
accumulation_steps = 1  # Adjust as needed
effective_batch_size = batch_size * accumulation_steps

'''best_val = float('inf')
patience, patience_counter = 5, 0

# Assuming dataset is already defined as your ImageFolder
total = len(dataset)
val_pct = 0.10  # for example, use 10% for validation

val_size = int(total * val_pct)
train_size = total - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])'''

train_losses = []

# training loop
for epoch in range(num_epochs):
    model.train()
    for i, (images, _) in enumerate(dataloader):
        images = images.to(device)
        loss = model(images)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch} | Step {i+1}/{len(dataloader)} | Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), f"diffusion_epoch_{epoch+1}.pt")

    # Clear memory
    del images, loss  # delete batch tensors
    gc.collect()
    torch.cuda.empty_cache()

    '''val_loss = validate(model, val_loader)
    print(f"Epoch {epoch}: val_loss = {val_loss:.4f}")
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break'''

print("Training complete!")

In [ ]:
!mkdir output

In [ ]:
'''# 2. Sampling (Insert Here)
model.eval()  # switch to eval mode for sampling
with torch.no_grad():  # no gradients needed
    sampled_imgs = model.sample(batch_size=4)
    torchvision.utils.save_image(
        sampled_imgs,
        'generated.png',
        normalize=True,
        value_range=(-1, 1)  # adjust if your model expects different range
    )

print("Sampled images saved to generated.png")
'''

from torchvision.utils import save_image

total = 160
batch = 16   # or higher, depending on GPU memory
runs = total // batch
extras = total % batch

for idx in range(runs):
    model.eval()
    with torch.no_grad():
        images = model.sample(batch_size=batch)
    save_image(images, f'output/batch_{idx}.png', nrow=4, normalize=True)

    # Clear memory between batches
    del images
    gc.collect()
    torch.cuda.empty_cache()

if extras:
    images = model.sample(batch_size=extras)
    save_image(images, f'output/batch_last.png', nrow=extras, normalize=True)

    del images
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
!pip install scikit-image

from skimage.metrics import peak_signal_noise_ratio as psnr_metric, structural_similarity as ssim_metric
import numpy as np
from PIL import Image

def calc_metrics(gt_path, gen_path):
    gt = np.array(Image.open(gt_path).convert('RGB'))
    gen = np.array(Image.open(gen_path).convert('RGB'))
    psnr = psnr_metric(gt, gen, data_range=255)
    ssim = ssim_metric(gt, gen, multichannel=True, data_range=255)
    return psnr, ssim

# Example usage
img = Image.open('filtered_images/1.jpg')
# Resize to 128×128
img_resized = img.resize((128, 128), resample=Image.BICUBIC)
# Save the resized image
img_resized.save("high_res_128.png")
gen = 'generated.png'
psnr, ssim = calc_metrics("high_res_128.png", "generated.png")
print(f"📏 PSNR: {psnr:.2f} dB, SSIM: {ssim:.4f}")


In [ ]:
!pip install paddleocr opencv-python-headless

from paddleocr import PaddleOCR
import cv2

ocr = PaddleOCR(use_angle_cls=True, lang='en')
def extract_text(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_OTSU)
    result = ocr.ocr(thresh, cls=True)
    return result[0][1][0] if result else ""

text = extract_text('generated.png')
print("Detected Text:", text)


In [ ]:
import re

province = {f"{i:02d}" for i in range(1, 82)}
forbidden = set()  # Add any disallowed combos here

def check_compliance(text):
    pattern = r'^(0[1-9]|[1-7][0-9]|8[0-1]) [A-Z]{1,3} \d{2,4}$'
    if re.match(pattern, text):
        return text.split()[0] in province and text not in forbidden
    return False

valid = check_compliance(text)
print("Compliance:", valid)


In [ ]:
# After sampling and saving 'generated.png'
psnr, ssim = calc_metrics('gt.png', 'generated.png')
text = extract_text('generated.png')
valid = check_compliance(text)

print(f"PSNR: {psnr:.2f}, SSIM: {ssim:.4f}, OCR Text: '{text}', Compliant: {valid}")


In [ ]:
import matplotlib.pyplot as plt

# plot the loss curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('VAE Loss over Time')
plt.legend()
plt.show()

In [ ]:
'''class CustomVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def encode(self, x):
      return self.encoder(x)

    def decode(self, z):
      return self.decoder(z)

    def forward(self, x):
      z = self.encode(x)
      x_reconstructed = self.decode(z)
      return x_reconstructed

    def load_pretrained_weights(self, weight_path):
      self.load_state_dict(torch.load(weight_path))

'''

import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomVAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def encode(self, x):
        mean, logvar = self.encoder(x)
        return mean, logvar

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mean, logvar = self.encode(x)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mean + eps * std
        z *= 0.18215  # scaling for latent space
        return self.decode(z), z


In [ ]:
'''from diffusers import AutoencoderKL

class DiffuserCompatibleVAE(AutoencoderKL):
    def __init__(self, vae):
        super().__init__()
        self.vae = vae

    def encode(self, x):
      mean, log_var = self.vae.encoder(x)
      print("mean")
      return mean, log_var

    def decode(self, z, **kwargs):
      print("Input shape:", z.shape)
      out = self.vae.decoder(z).unsqueeze(0)
      print(out.shape)
      return out
'''

from diffusers.models.modeling_utils import ModelMixin
from torch.distributions import Normal

class DiffuserCompatibleVAE(ModelMixin):
    def __init__(self, custom_vae):
        super().__init__()
        self.vae = custom_vae

        # Required for SD pipeline initialization
        self.config = type("Config", (), {
            "scaling_factor": 0.18215,
            "block_out_channels": [32, 64, 64]  # Adjust based on your encoder’s structure
        })()

    def encode(self, x, **kwargs):
        mean, logvar = self.vae.encode(x)
        std = torch.exp(0.5 * logvar)
        return {
            "latent_dist": Normal(mean, std)
        }

    def decode(self, z, return_dict=True, **kwargs):
        x = self.vae.decode(z)
        if return_dict:
            return {"sample": x}
        return (x,)



In [ ]:
from huggingface_hub import snapshot_download

snapshot_download("CompVis/stable-diffusion-v1-4", local_dir="./model")

In [ ]:
from diffusers import AutoencoderKL
AutoencoderKL.from_pretrained("./model/vae")

In [ ]:
import torch

weight = torch.load("./vae_model_epoch_1.pth", map_location="cpu")

new_weights = {}
for k, v in weight.items():
  new_key = k.replace("_", "")

  new_key = new_key.replace("residuallayer", "residual_layer").replace("inproj", "in_proj").replace("outproj", "out_proj")
  new_weights[new_key] = v

encoder = Encoder()
decoder = Decoder()

vae = CustomVAE(encoder, decoder)
vae.load_state_dict(new_weights)

device = torch.device("cuda")
vae = vae.to(device)

In [ ]:
vae

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

compatible_vae = DiffuserCompatibleVAE(vae)
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4",
    vae=compatible_vae,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

In [ ]:
prompt = (
    "a photo of a modern car parked on the street with a visible license plate, "
    "taken from the rear, realistic lighting, natural outdoor setting, "
    "photorealistic, ultra high resolution, sharp details, cinematic"
)

'''prompt = (
    "a photo of Turkish license plate, 01 A 2808"
    "photorealistic, ultra high resolution, sharp details, cinematic"
)'''

image = pipe(prompt, num_inference_steps=50).images[0]

In [ ]:
image